这一章可以分成两大块：

第一块是 Go 里的设计模式。
原来 Java 面试里讲设计模式，常常会围绕类、继承、接口、抽象类、Spring AOP、工厂、代理这些概念展开。Go 不一样。Go 没有类继承，没有抽象类，也不鼓励为了模式而模式。Go 更强调组合、接口、函数、一等函数、闭包、结构体嵌入和小接口。

所以 Go 里的设计模式回答，不应该把 Java 那套类图硬搬过来，而应该这样说：

> Go 也可以实现传统设计模式，但表达方式更轻量。很多模式在 Go 里不一定需要复杂类层次，而是通过 interface、struct、函数类型、闭包、组合和 middleware 链来实现。

第二块是数据结构与算法。
这部分和语言关系没那么强，但 Go 有自己的实现习惯。比如数组链表题里，数组更多用 slice；栈和队列通常也用 slice；堆使用 container/heap；哈希表用 map；排序可以用 sort.Ints、sort.Slice，也可以手写快排、归并、堆排；链表题要自己定义 ListNode；树题要自己定义 TreeNode。

这章的目标不是把所有算法题都写完，而是形成面试回答框架：

* 复杂度怎么讲
* 数组和链表怎么选
* 树、堆、哈希分别解决什么问题
* 排序算法怎么分类
* 栈、队列、单调栈、单调队列怎么理解
* Go 里常见数据结构怎么实现
* 面试被追问时怎么把“结构特征、复杂度、场景、题型”一起说清楚

一句话总结：

> Go 版设计模式重点是“少继承、多组合、小接口、函数化”；Go 版数据结构算法重点是“slice、map、container/heap、sort 包，以及力扣常见模板”。

---

# 一、Go 里的设计模式怎么理解

## 【文本块 2】Q：Go 里还需要设计模式吗？

需要，但不要机械套 Java 的设计模式。

设计模式本质上是对常见代码组织问题的经验总结。它解决的是：

* 如何隔离变化
* 如何降低耦合
* 如何提高扩展性
* 如何复用公共逻辑
* 如何让代码更容易测试和维护

这些问题在 Go 里同样存在。

但 Go 的语言特性和 Java 不一样。Java 里很多模式依赖类继承、抽象类、接口层次；Go 更倾向于：

* 用 interface 表达行为
* 用 struct 表达数据
* 用组合代替继承
* 用函数类型代替策略对象
* 用闭包保存上下文
* 用 middleware 链实现装饰和拦截
* 用 sync.Once 实现单例初始化
* 用 channel 或事件总线做观察者

面试里可以这样回答：

> Go 里当然可以用设计模式，但不应该为了模式写很多抽象层。Go 更鼓励小接口、组合和函数式写法。很多传统模式在 Go 里会被简化，比如策略模式可以直接用函数类型实现，单例模式通常用 sync.Once，装饰器和责任链常常体现为 HTTP middleware 或 gRPC interceptor。

---

# 二、策略模式

## 【文本块 3】Q：什么是策略模式？

策略模式的核心思想是：

> 把一组可互相替换的算法或行为抽象出来，在运行时按需选择。

它解决的问题是：同一件事有多种实现，不希望在主流程里写一堆 if-else 或 switch。

典型场景包括：

* 不同支付方式：支付宝、微信、银行卡
* 不同优惠计算：满减、折扣、会员价
* 不同路由规则：按用户、按地区、按权重
* 不同排序规则：按价格、销量、评分
* 不同风控策略：黑名单、频率、设备指纹
* 不同消息发送方式：短信、邮件、站内信

在 Go 里，策略模式有两种常见写法。

第一种，用 interface。
适合策略本身比较复杂，有多个方法，或者需要保存内部状态。

第二种，用函数类型。
适合策略逻辑很轻量，只有一个核心动作。

---

## 【代码块 1】策略模式：interface 写法

```go
package main

import (
	"context"
	"fmt"
)

type PayStrategy interface {
	Pay(ctx context.Context, amount int64) error
}

type WeChatPay struct{}

func (p *WeChatPay) Pay(ctx context.Context, amount int64) error {
	fmt.Println("pay by WeChat:", amount)
	return nil
}

type AliPay struct{}

func (p *AliPay) Pay(ctx context.Context, amount int64) error {
	fmt.Println("pay by AliPay:", amount)
	return nil
}

type PaymentService struct {
	strategies map[string]PayStrategy
}

func NewPaymentService() *PaymentService {
	return &PaymentService{
		strategies: map[string]PayStrategy{
			"wechat": &WeChatPay{},
			"alipay": &AliPay{},
		},
	}
}

func (s *PaymentService) Pay(ctx context.Context, payType string, amount int64) error {
	strategy, ok := s.strategies[payType]
	if !ok {
		return fmt.Errorf("unsupported pay type: %s", payType)
	}
	return strategy.Pay(ctx, amount)
}

func main() {
	svc := NewPaymentService()

	_ = svc.Pay(context.Background(), "wechat", 100)
	_ = svc.Pay(context.Background(), "alipay", 200)
}
```

---

## 【文本块 4】代码解释

这里的 `PayStrategy` 就是策略接口。

`WeChatPay` 和 `AliPay` 是两种具体策略。

`PaymentService` 不关心具体支付方式怎么实现，只负责根据 payType 找到对应策略，然后调用统一的 `Pay` 方法。

相比在 `Pay` 方法里写：

```go
if payType == "wechat" {
    ...
} else if payType == "alipay" {
    ...
}
```

策略模式的好处是：

* 新增策略时不需要大改主流程
* 每种策略可以单独测试
* 主流程更稳定
* 符合开闭原则
* 适合策略数量较多且经常扩展的场景

---

## 【代码块 2】策略模式：函数类型写法

```go
package main

import "fmt"

type DiscountFunc func(price int64) int64

func FullReduction(price int64) int64 {
	if price >= 1000 {
		return price - 100
	}
	return price
}

func PercentageDiscount(price int64) int64 {
	return price * 80 / 100
}

type PriceService struct {
	discounts map[string]DiscountFunc
}

func NewPriceService() *PriceService {
	return &PriceService{
		discounts: map[string]DiscountFunc{
			"full_reduction": FullReduction,
			"percentage":     PercentageDiscount,
		},
	}
}

func (s *PriceService) Calculate(strategy string, price int64) (int64, error) {
	fn, ok := s.discounts[strategy]
	if !ok {
		return 0, fmt.Errorf("unsupported discount strategy: %s", strategy)
	}
	return fn(price), nil
}

func main() {
	svc := NewPriceService()

	p1, _ := svc.Calculate("full_reduction", 1200)
	p2, _ := svc.Calculate("percentage", 1200)

	fmt.Println(p1)
	fmt.Println(p2)
}
```

---

## 【文本块 5】追问：Go 里策略模式用 interface 还是函数类型？

看复杂度。

如果策略只是“输入参数 -> 输出结果”的一段简单逻辑，用函数类型更轻。

比如：

* 折扣计算
* 排序比较函数
* 简单校验规则
* 路由 hash 函数

如果策略需要多个方法、内部依赖、状态管理、生命周期管理，用 interface 更合适。

比如：

* 支付通道
* 存储后端
* 消息队列 producer
* 第三方 API client
* 风控策略引擎

面试里可以这样回答：

> Go 里策略模式不一定要写一堆类。简单策略可以用函数类型，复杂策略用 interface。核心不是形式，而是把可替换行为从主流程中解耦出来。

---

# 三、单例模式

## 【文本块 6】Q：Go 里怎么实现单例模式？

单例模式的目标是：

> 保证一个对象在进程内只初始化一次，并提供全局访问点。

常见场景：

* 配置对象
* 日志对象
* 数据库连接池
* Redis client
* 监控 reporter
* 全局缓存
* ID 生成器

在 Go 里最推荐使用 `sync.Once` 实现单例初始化。

它能保证即使多个 goroutine 同时调用，也只执行一次初始化函数。

---

## 【代码块 3】sync.Once 实现单例

```go
package main

import (
	"fmt"
	"sync"
)

type Config struct {
	AppName string
	Env     string
}

var (
	config     *Config
	configOnce sync.Once
)

func GetConfig() *Config {
	configOnce.Do(func() {
		fmt.Println("init config once")
		config = &Config{
			AppName: "go-backend",
			Env:     "dev",
		}
	})
	return config
}

func main() {
	c1 := GetConfig()
	c2 := GetConfig()

	fmt.Println(c1 == c2)
	fmt.Println(c1)
}
```

---

## 【文本块 7】代码解释

`sync.Once` 的语义是：传入 `Do` 的函数只会被执行一次。

即使有多个 goroutine 同时执行：

```go
GetConfig()
```

初始化逻辑也只会执行一次。

这比自己写双重检查锁更稳，因为 `sync.Once` 已经帮我们处理了并发安全和内存可见性问题。

---

## 【文本块 8】追问：Go 里要不要大量使用单例？

不要滥用。

单例虽然方便，但会带来几个问题：

第一，隐藏依赖。
函数内部直接调用全局单例，外部看不出它依赖了什么。

第二，测试困难。
单例对象不好替换，单元测试 mock 比较麻烦。

第三，全局状态污染。
多个测试或多个业务逻辑共享一个全局对象，容易互相影响。

第四，初始化顺序复杂。
全局对象之间互相依赖时，排查问题会很痛苦。

Go 工程里更推荐：

* 基础设施对象可以在 main 里初始化一次
* 通过构造函数注入到 service 中
* 业务代码依赖 interface，而不是直接依赖全局变量
* sync.Once 适合懒加载或全局基础组件，但不要把所有东西都做成单例

面试里可以这样说：

> Go 里实现单例通常用 sync.Once，但我不会滥用全局单例。更推荐在 main 初始化依赖，然后通过构造函数注入到 service。这样依赖关系清晰，也更容易测试。

---

# 四、观察者模式

## 【文本块 9】Q：什么是观察者模式？

观察者模式的核心是：

> 一个对象状态发生变化后，自动通知一组依赖它的对象。

它常用于事件发布和订阅。

典型场景：

* 订单创建后通知库存、积分、消息服务
* 用户注册后发送欢迎邮件
* 配置变更后通知各模块 reload
* 任务完成后触发多个后续动作
* 领域事件 Domain Event
* 本地事件总线
* MQ 发布订阅

观察者模式本质上是让主流程只关心核心动作，把附加动作交给监听者扩展。

例如，订单创建主流程只负责创建订单。
至于发通知、发积分、写日志、同步 ES，可以作为事件监听者处理。

---

## 【代码块 4】观察者模式：本地事件总线

```go
package main

import (
	"context"
	"fmt"
	"sync"
)

type Event interface {
	Name() string
}

type OrderCreatedEvent struct {
	OrderID int64
	UserID  int64
}

func (e OrderCreatedEvent) Name() string {
	return "order.created"
}

type Handler func(ctx context.Context, event Event) error

type EventBus struct {
	mu       sync.RWMutex
	handlers map[string][]Handler
}

func NewEventBus() *EventBus {
	return &EventBus{
		handlers: make(map[string][]Handler),
	}
}

func (b *EventBus) Subscribe(eventName string, handler Handler) {
	b.mu.Lock()
	defer b.mu.Unlock()

	b.handlers[eventName] = append(b.handlers[eventName], handler)
}

func (b *EventBus) Publish(ctx context.Context, event Event) error {
	b.mu.RLock()
	handlers := append([]Handler(nil), b.handlers[event.Name()]...)
	b.mu.RUnlock()

	for _, handler := range handlers {
		if err := handler(ctx, event); err != nil {
			return fmt.Errorf("handle event %s: %w", event.Name(), err)
		}
	}

	return nil
}

func main() {
	bus := NewEventBus()

	bus.Subscribe("order.created", func(ctx context.Context, event Event) error {
		e := event.(OrderCreatedEvent)
		fmt.Println("send notification for order:", e.OrderID)
		return nil
	})

	bus.Subscribe("order.created", func(ctx context.Context, event Event) error {
		e := event.(OrderCreatedEvent)
		fmt.Println("add points for user:", e.UserID)
		return nil
	})

	_ = bus.Publish(context.Background(), OrderCreatedEvent{
		OrderID: 1001,
		UserID:  2002,
	})
}
```

---

## 【文本块 10】代码解释

这里的 EventBus 就是一个简单观察者模式实现。

`Subscribe` 用于注册监听者。
`Publish` 用于发布事件。
同一个事件可以有多个 handler。

优点是主流程和附加逻辑解耦。

缺点是：

* handler 失败怎么处理要设计
* 同步发布会增加主链路耗时
* 异步发布要考虑 goroutine 泄漏
* 事件顺序和事务一致性要设计
* 本地事件只在单进程内有效，跨服务要用 MQ

面试里可以说：

> 观察者模式在单进程内可以用事件总线实现，跨服务场景通常会演进成 MQ 发布订阅。核心思想都是让主流程发布事件，由订阅者独立扩展后续动作。

---

# 五、工厂模式

## 【文本块 11】Q：Go 里怎么理解工厂模式？

工厂模式的核心是：

> 把对象创建逻辑封装起来，让调用方不直接依赖具体实现。

Go 里工厂模式很常见，只是大家不一定叫它工厂模式。

比如：

```go
NewUserService(...)
NewRedisClient(...)
NewOrderRepository(...)
NewHTTPServer(...)
```

这些 `NewXxx` 函数本质上就是构造函数或简单工厂。

适用场景：

* 创建逻辑复杂
* 需要校验配置
* 需要注入依赖
* 需要根据类型选择不同实现
* 希望调用方只依赖接口

---

## 【代码块 5】简单工厂：根据类型创建存储实现

```go
package main

import (
	"fmt"
)

type Storage interface {
	Save(key string, value []byte) error
}

type LocalStorage struct{}

func (s *LocalStorage) Save(key string, value []byte) error {
	fmt.Println("save to local:", key)
	return nil
}

type S3Storage struct{}

func (s *S3Storage) Save(key string, value []byte) error {
	fmt.Println("save to s3:", key)
	return nil
}

func NewStorage(kind string) (Storage, error) {
	switch kind {
	case "local":
		return &LocalStorage{}, nil
	case "s3":
		return &S3Storage{}, nil
	default:
		return nil, fmt.Errorf("unsupported storage kind: %s", kind)
	}
}

func main() {
	storage, _ := NewStorage("local")
	_ = storage.Save("avatar.png", []byte("fake image"))
}
```

---

## 【文本块 12】追问：Go 里构造函数为什么一般返回接口还是结构体？

这要看语义。

如果调用方需要依赖抽象，并且你希望隐藏具体实现，可以返回接口：

```go
func NewStorage(kind string) (Storage, error)
```

如果调用方需要访问结构体自己的方法，或者这个类型本身就是明确的实现，可以返回结构体指针：

```go
func NewOrderService(repo OrderRepo) *OrderService
```

Go 里不建议为了“面向接口”而所有构造函数都返回接口。

更常见的原则是：

> 接口由使用方定义，结构体由实现方返回。

也就是说，如果没有多个实现，没有替换需求，没有测试隔离需求，直接返回结构体也很好。

---

# 六、Functional Options 模式

## 【文本块 13】Q：什么是 Functional Options 模式？

Functional Options 是 Go 里非常常见的构造配置模式。

它解决的是：

* 构造参数很多
* 参数有默认值
* 未来可能继续扩展
* 不想写一堆构造函数
* 不想让调用方直接修改内部字段

常见写法是：

```go
type Option func(*Server)

func WithTimeout(...) Option
func WithLogger(...) Option
func NewServer(opts ...Option) *Server
```

这比下面这种写法更灵活：

```go
NewServer(addr string, timeout time.Duration, maxConn int, logger Logger, ...)
```

参数太多时，可读性很差，也容易传错。

---

## 【代码块 6】Functional Options 示例

```go
package main

import (
	"fmt"
	"time"
)

type Server struct {
	addr    string
	timeout time.Duration
	maxConn int
}

type Option func(*Server)

func WithAddr(addr string) Option {
	return func(s *Server) {
		s.addr = addr
	}
}

func WithTimeout(timeout time.Duration) Option {
	return func(s *Server) {
		s.timeout = timeout
	}
}

func WithMaxConn(maxConn int) Option {
	return func(s *Server) {
		s.maxConn = maxConn
	}
}

func NewServer(opts ...Option) *Server {
	s := &Server{
		addr:    ":8080",
		timeout: 3 * time.Second,
		maxConn: 1000,
	}

	for _, opt := range opts {
		opt(s)
	}

	return s
}

func main() {
	s := NewServer(
		WithAddr(":9090"),
		WithTimeout(5*time.Second),
	)

	fmt.Printf("%+v\n", s)
}
```

---

## 【文本块 14】代码解释

`NewServer` 先设置默认值，再依次应用 options。

调用方只传自己想改的配置：

```go
NewServer(WithAddr(":9090"))
```

而不是必须传一堆无关参数。

很多 Go 库都会用这种模式，比如 HTTP server、RPC client、缓存组件、SDK client 等。

面试里可以说：

> Functional Options 是 Go 里很典型的工程模式，适合构造参数多且有默认值的对象。它比多参数构造函数更可读，也比暴露配置结构体更容易控制初始化逻辑。

---

# 七、装饰器模式与 Middleware

## 【文本块 15】Q：Go 里装饰器模式常见在哪里？

Go 里最常见的装饰器模式就是 middleware。

装饰器模式的核心是：

> 不修改原对象代码，在外面包一层，为它增加额外能力。

HTTP middleware 就是典型例子。

原始 handler 只处理业务。
middleware 在它外面包一层，可以增加：

* 日志
* 鉴权
* recover
* 限流
* trace
* metrics
* CORS
* timeout

这比在每个 handler 里都写一遍这些逻辑好很多。

---

## 【代码块 7】HTTP Middleware 装饰器模式

```go
package main

import (
	"log"
	"net/http"
	"time"
)

type Middleware func(http.Handler) http.Handler

func Logging() Middleware {
	return func(next http.Handler) http.Handler {
		return http.HandlerFunc(func(w http.ResponseWriter, r *http.Request) {
			start := time.Now()
			next.ServeHTTP(w, r)
			log.Printf("method=%s path=%s cost=%s", r.Method, r.URL.Path, time.Since(start))
		})
	}
}

func Recover() Middleware {
	return func(next http.Handler) http.Handler {
		return http.HandlerFunc(func(w http.ResponseWriter, r *http.Request) {
			defer func() {
				if rec := recover(); rec != nil {
					log.Printf("panic: %v", rec)
					http.Error(w, "internal server error", http.StatusInternalServerError)
				}
			}()

			next.ServeHTTP(w, r)
		})
	}
}

func Chain(h http.Handler, middlewares ...Middleware) http.Handler {
	for i := len(middlewares) - 1; i >= 0; i-- {
		h = middlewares[i](h)
	}
	return h
}

func main() {
	mux := http.NewServeMux()

	mux.HandleFunc("/hello", func(w http.ResponseWriter, r *http.Request) {
		_, _ = w.Write([]byte("hello"))
	})

	handler := Chain(mux, Recover(), Logging())

	_ = http.ListenAndServe(":8080", handler)
}
```

---

## 【文本块 16】代码解释

这里 `Logging` 和 `Recover` 都没有修改业务 handler。

它们只是把原始 handler 包起来，在前后增加逻辑。

这就是 Go Web 框架中 middleware 的基本原理。

面试里可以这样答：

> Go 里的装饰器模式最常见就是 middleware 或 interceptor。它通过函数包装给原始处理逻辑增加日志、鉴权、限流、trace 等横切能力，避免这些逻辑散落在业务代码里。

---

# 八、适配器模式

## 【文本块 17】Q：什么是适配器模式？Go 里怎么用？

适配器模式的核心是：

> 把一个已有对象或第三方接口，转换成当前系统期望的接口。

典型场景：

* 第三方支付 SDK 的接口不符合自己系统的 Payment 接口
* 老系统 API 和新系统接口不一致
* 不同云存储 SDK 统一成 Storage 接口
* 不同消息队列 client 统一成 Producer 接口
* 测试时把真实 client 替换成 fake client

Go 里适配器模式非常自然，因为 interface 是隐式实现的。

只要一个结构体实现了目标接口的方法，就自动满足接口。

---

## 【代码块 8】适配器模式：统一第三方短信 SDK

```go
package main

import (
	"context"
	"fmt"
)

type SMSClient interface {
	Send(ctx context.Context, phone string, content string) error
}

// 第三方 SDK，接口不受我们控制。
type ThirdPartySMS struct{}

func (s *ThirdPartySMS) PushMessage(mobile string, text string) error {
	fmt.Println("third party sms:", mobile, text)
	return nil
}

// 适配器，把 ThirdPartySMS 适配成我们系统里的 SMSClient。
type ThirdPartySMSAdapter struct {
	client *ThirdPartySMS
}

func NewThirdPartySMSAdapter(client *ThirdPartySMS) *ThirdPartySMSAdapter {
	return &ThirdPartySMSAdapter{client: client}
}

func (a *ThirdPartySMSAdapter) Send(ctx context.Context, phone string, content string) error {
	return a.client.PushMessage(phone, content)
}

func Notify(ctx context.Context, client SMSClient, phone string) error {
	return client.Send(ctx, phone, "your order has been created")
}

func main() {
	adapter := NewThirdPartySMSAdapter(&ThirdPartySMS{})

	_ = Notify(context.Background(), adapter, "13800000000")
}
```

---

## 【文本块 18】工程回答

适配器模式在 Go 里非常实用。

因为很多真实项目都会遇到：

* 第三方 SDK 不好用
* 多个供应商接口不同
* 业务层不想直接依赖 SDK
* 单元测试需要替换外部依赖

面试里可以说：

> 我会在 infra 层写 adapter，把第三方 SDK 转成业务层定义的 interface。业务代码只依赖自己的接口，不直接依赖第三方 SDK，这样后续切供应商或做 mock 都比较容易。

---

# 九、依赖注入

## 【文本块 19】Q：Go 里怎么做依赖注入？

Go 没有 Spring 那种大型 IOC 容器是正常的。

Go 更常见的依赖注入方式是：

* 构造函数注入
* interface 注入
* main 函数组装依赖
* wire 这类代码生成工具
* fx 这类运行时 DI 框架

普通项目里，构造函数注入已经足够。

比如：

```go
func NewOrderService(repo OrderRepo, producer Producer) *OrderService
```

这样 OrderService 的依赖很清楚，也方便测试。

---

## 【代码块 9】构造函数注入

```go
package main

import (
	"context"
	"fmt"
)

type OrderRepo interface {
	Create(ctx context.Context, orderID int64) error
}

type Producer interface {
	Send(ctx context.Context, topic string, body []byte) error
}

type OrderService struct {
	repo     OrderRepo
	producer Producer
}

func NewOrderService(repo OrderRepo, producer Producer) *OrderService {
	return &OrderService{
		repo:     repo,
		producer: producer,
	}
}

func (s *OrderService) CreateOrder(ctx context.Context, orderID int64) error {
	if err := s.repo.Create(ctx, orderID); err != nil {
		return fmt.Errorf("create order: %w", err)
	}

	if err := s.producer.Send(ctx, "order.created", []byte(fmt.Sprint(orderID))); err != nil {
		return fmt.Errorf("send order event: %w", err)
	}

	return nil
}
```

---

## 【文本块 20】追问：Go 为什么不太喜欢复杂 IOC？

因为 Go 更强调显式。

复杂 IOC 容器虽然能自动装配依赖，但也可能带来：

* 依赖关系不清晰
* 初始化顺序隐蔽
* 运行时报错
* 代码跳转困难
* 测试替换复杂
* 新人理解成本高

Go 项目更常见的风格是：

> 在 main 或 wire 生成代码里把依赖组装清楚，业务结构体通过构造函数接收依赖。

这样虽然多写一些代码，但依赖关系非常直接。

---

# 十、设计模式速记版

## 【文本块 21】设计模式 Go 版速记

策略模式：
把可替换算法抽象出来。Go 里简单策略用函数类型，复杂策略用 interface。

单例模式：
保证对象只初始化一次。Go 里推荐 sync.Once，但不要滥用全局单例。

观察者模式：
状态变化后通知多个监听者。单进程内可以用事件总线，跨服务通常用 MQ。

工厂模式：
封装对象创建逻辑。Go 里常见就是 NewXxx 函数。

Functional Options：
解决构造参数多、有默认值、未来可扩展的问题。

装饰器模式：
不改原逻辑，外面包一层增强能力。Go 里最常见是 middleware/interceptor。

适配器模式：
把第三方 SDK 或旧接口适配成系统内部统一接口。

依赖注入：
Go 里优先构造函数注入，业务依赖 interface，main 负责组装。

一句话总结：

> Go 里的设计模式不要背类图，要看它解决什么工程问题。能用函数就不用复杂对象，能用组合就不要继承，能显式注入就不要隐藏全局依赖。

---

# 十一、复杂度基础

## 【文本块 22】Q：时间复杂度和空间复杂度怎么理解？

时间复杂度表示：

> 随着输入规模 n 增大，算法执行次数大致如何增长。

空间复杂度表示：

> 随着输入规模 n 增大，算法额外占用的存储空间大致如何增长。

它们关注的是增长趋势，不是某一次运行具体花了多少毫秒。

常见复杂度从低到高：

* O(1)：常数复杂度
* O(log n)：对数复杂度
* O(n)：线性复杂度
* O(n log n)：线性对数复杂度
* O(n²)：平方复杂度
* O(2ⁿ)：指数复杂度

面试里不要只背定义，要能结合例子：

* 数组下标访问是 O(1)
* 二分查找是 O(log n)
* 遍历数组是 O(n)
* 快排平均是 O(n log n)
* 两层嵌套循环常见是 O(n²)

---

## 【代码块 10】复杂度示例

```go
package main

import "fmt"

func O1(nums []int) int {
	return nums[0]
}

func On(nums []int) int {
	sum := 0
	for _, x := range nums {
		sum += x
	}
	return sum
}

func On2(nums []int) int {
	count := 0
	for i := range nums {
		for j := range nums {
			if nums[i] == nums[j] {
				count++
			}
		}
	}
	return count
}

func main() {
	nums := []int{1, 2, 3, 4}

	fmt.Println(O1(nums))
	fmt.Println(On(nums))
	fmt.Println(On2(nums))
}
```

---

## 【文本块 23】追问：为什么复杂度要忽略常数？

因为复杂度看的是输入规模趋近很大时的增长趋势。

比如：

```text
100n
n²
```

当 n 很小时，100n 可能比 n² 大。
但当 n 很大时，n² 增长会远远超过 100n。

所以复杂度会忽略常数和低阶项。

例如：

```text
3n² + 100n + 1000
```

最终记作：

```text
O(n²)
```

但工程里也不能完全无视常数。
在高性能场景下，常数、缓存局部性、内存分配、锁竞争都会影响真实表现。

面试里可以说：

> 算法复杂度用来判断增长趋势，但工程性能还要看常数、内存分配、缓存命中、并发竞争和 IO 等因素。

---

# 十二、数组、slice 与链表

## 【文本块 24】Q：数组和链表有什么区别？

数组的核心特点是连续内存。

优点：

* 支持下标随机访问
* 访问复杂度 O(1)
* 缓存局部性好
* 遍历速度快

缺点：

* 中间插入删除需要移动元素
* 动态扩容可能需要复制

链表的核心特点是节点不连续，通过指针连接。

优点：

* 已知节点时插入删除是 O(1)
* 不需要连续大块内存

缺点：

* 不支持随机访问
* 查找第 i 个节点需要 O(n)
* 每个节点有额外指针开销
* 缓存局部性差

Go 里平时更常用 slice，而不是链表。

原因是：

* slice 简单
* 连续内存遍历快
* 标准库和生态都围绕 slice
* 链表节点分散，实际性能未必好
* 力扣链表题更多是算法训练，不代表业务代码常用链表

---

## 【代码块 11】Go 里 slice 插入和删除

```go
package main

import "fmt"

func Insert(nums []int, index int, value int) []int {
	nums = append(nums, 0)
	copy(nums[index+1:], nums[index:])
	nums[index] = value
	return nums
}

func Delete(nums []int, index int) []int {
	copy(nums[index:], nums[index+1:])
	return nums[:len(nums)-1]
}

func main() {
	nums := []int{1, 2, 4, 5}

	nums = Insert(nums, 2, 3)
	fmt.Println(nums)

	nums = Delete(nums, 1)
	fmt.Println(nums)
}
```

---

## 【文本块 25】代码解释

slice 中间插入时，需要把 index 后面的元素整体后移。

slice 中间删除时，需要把 index 后面的元素整体前移。

所以插入删除本质上是 O(n)。

但是因为 slice 是连续内存，CPU 缓存友好，很多业务场景下仍然比链表快。

面试里可以说：

> 不要机械说链表增删一定比数组快。链表只有在已经拿到节点时插删才是 O(1)，如果还要先遍历定位，整体仍然是 O(n)。实际业务里 slice 因为连续内存和缓存局部性，往往更常用。

---

## 【代码块 12】链表节点定义与反转链表

```go
package main

import "fmt"

type ListNode struct {
	Val  int
	Next *ListNode
}

func ReverseList(head *ListNode) *ListNode {
	var prev *ListNode
	cur := head

	for cur != nil {
		next := cur.Next
		cur.Next = prev
		prev = cur
		cur = next
	}

	return prev
}

func PrintList(head *ListNode) {
	for head != nil {
		fmt.Print(head.Val, " ")
		head = head.Next
	}
	fmt.Println()
}

func main() {
	head := &ListNode{Val: 1}
	head.Next = &ListNode{Val: 2}
	head.Next.Next = &ListNode{Val: 3}

	PrintList(head)

	newHead := ReverseList(head)
	PrintList(newHead)
}
```

---

## 【文本块 26】链表题高频套路

链表题其实套路非常固定：

第一，反转链表。
核心是保存 next，然后改变 cur.Next 指向。

第二，快慢指针。
常用于找中点、判断环、找环入口。

第三，虚拟头节点 dummy。
常用于删除节点、合并链表、处理头节点可能变化的问题。

第四，双指针间隔。
常用于删除倒数第 k 个节点。

第五，递归。
常用于反转链表、合并链表、回溯式处理。

面试里可以说：

> 链表题难点不在数据结构本身，而在指针更新顺序。写代码时一定先保存 next，再改指针，最后移动 cur。

---

# 十三、二分查找

## 【文本块 27】Q：二分查找为什么容易写错？

因为二分真正难的不是“每次砍一半”，而是边界。

二分查找的前提是：

> 搜索空间具有单调性。

常见二分类型：

第一，找某个值是否存在。
第二，找左边界。
第三，找右边界。
第四，二分答案。

二分答案的本质是：

> 在一个有单调性的答案空间里，找最小可行值或最大可行值。

比如：

* 最小运载能力
* 最小吃香蕉速度
* 最小满足条件时间
* 最大可行长度

---

## 【代码块 13】二分查找：找目标值

```go
package main

import "fmt"

func BinarySearch(nums []int, target int) int {
	left, right := 0, len(nums)-1

	for left <= right {
		mid := left + (right-left)/2

		if nums[mid] == target {
			return mid
		}

		if nums[mid] < target {
			left = mid + 1
		} else {
			right = mid - 1
		}
	}

	return -1
}

func main() {
	fmt.Println(BinarySearch([]int{1, 3, 5, 7, 9}, 7))
}
```

---

## 【代码块 14】二分查找：找左边界

```go
package main

import "fmt"

func LowerBound(nums []int, target int) int {
	left, right := 0, len(nums)

	for left < right {
		mid := left + (right-left)/2

		if nums[mid] >= target {
			right = mid
		} else {
			left = mid + 1
		}
	}

	return left
}

func main() {
	nums := []int{1, 2, 2, 2, 3, 4}

	fmt.Println(LowerBound(nums, 2))
	fmt.Println(LowerBound(nums, 5))
}
```

---

## 【文本块 28】二分模板怎么记？

推荐记半开区间 `[left, right)` 模板。

初始：

```go
left, right := 0, len(nums)
```

循环：

```go
for left < right
```

收缩：

```go
right = mid
left = mid + 1
```

这个模板适合找左边界，也适合很多二分答案题。

面试里可以说：

> 我一般用半开区间写二分，核心是不让答案区间丢失。每次判断 mid 是否满足条件，满足就尝试收缩右边界，不满足就移动左边界。

---

# 十四、树

## 【文本块 29】Q：树结构解决什么问题？

树解决的是层级关系和递归结构问题。

常见关键词：

* 根节点
* 叶子节点
* 父节点
* 子节点
* 左子树
* 右子树
* 深度
* 高度
* 路径
* 祖先
* 最近公共祖先

面试里常见的是二叉树。

一看到这些词，就要进入树的思路：

* 左右子树
* 递归
* DFS
* BFS
* 前序
* 中序
* 后序
* 层序遍历

二叉搜索树 BST 的特点是：

> 左子树都小于当前节点，右子树都大于当前节点，中序遍历有序。

堆的底层也可以看成完全二叉树。

---

## 【代码块 15】二叉树定义与前序遍历

```go
package main

import "fmt"

type TreeNode struct {
	Val   int
	Left  *TreeNode
	Right *TreeNode
}

func Preorder(root *TreeNode, result *[]int) {
	if root == nil {
		return
	}

	*result = append(*result, root.Val)
	Preorder(root.Left, result)
	Preorder(root.Right, result)
}

func main() {
	root := &TreeNode{Val: 1}
	root.Left = &TreeNode{Val: 2}
	root.Right = &TreeNode{Val: 3}

	var result []int
	Preorder(root, &result)

	fmt.Println(result)
}
```

---

## 【文本块 30】树的三种 DFS 遍历

前序遍历：

```text
根 -> 左 -> 右
```

适合先处理当前节点，再处理子树。

中序遍历：

```text
左 -> 根 -> 右
```

二叉搜索树中序遍历是有序的。

后序遍历：

```text
左 -> 右 -> 根
```

适合先收集子树信息，再处理当前节点。比如求树高度、判断平衡树、删除树节点等。

面试里可以说：

> 树题很多都可以用递归思考。前序强调自顶向下，后序强调自底向上，中序在 BST 中有特殊意义。

---

## 【代码块 16】二叉树层序遍历 BFS

```go
package main

import "fmt"

type TreeNode struct {
	Val   int
	Left  *TreeNode
	Right *TreeNode
}

func LevelOrder(root *TreeNode) [][]int {
	if root == nil {
		return nil
	}

	var result [][]int
	queue := []*TreeNode{root}

	for len(queue) > 0 {
		size := len(queue)
		level := make([]int, 0, size)

		for i := 0; i < size; i++ {
			node := queue[0]
			queue = queue[1:]

			level = append(level, node.Val)

			if node.Left != nil {
				queue = append(queue, node.Left)
			}
			if node.Right != nil {
				queue = append(queue, node.Right)
			}
		}

		result = append(result, level)
	}

	return result
}

func main() {
	root := &TreeNode{Val: 1}
	root.Left = &TreeNode{Val: 2}
	root.Right = &TreeNode{Val: 3}
	root.Left.Left = &TreeNode{Val: 4}

	fmt.Println(LevelOrder(root))
}
```

---

## 【文本块 31】追问：DFS 和 BFS 怎么选？

DFS 适合：

* 找路径
* 回溯
* 树的递归处理
* 连通块搜索
* 需要走到底再回退

BFS 适合：

* 最短路径
* 层序遍历
* 距离扩散
* 最少步数
* 按层处理

在二叉树里：

* 前序、中序、后序属于 DFS
* 层序遍历属于 BFS

面试里可以说：

> DFS 更像一路走到底，适合路径和递归结构；BFS 更像一圈一圈扩散，适合最短路径和层级问题。

---

# 十五、堆

## 【文本块 32】Q：堆是什么？适合解决什么问题？

堆是一种特殊的完全二叉树。

常见有两种：

小根堆：堆顶是最小值。
大根堆：堆顶是最大值。

堆最擅长解决：

* 快速拿最小值
* 快速拿最大值
* TopK
* 优先队列
* 合并 K 个有序链表
* 数据流中位数
* 定时任务调度

堆的核心复杂度：

* 获取堆顶：O(1)
* 插入元素：O(log n)
* 删除堆顶：O(log n)
* 建堆：O(n)

Go 标准库提供 `container/heap`，但使用起来需要实现接口。

---

## 【代码块 17】Go 小根堆实现 TopK

```go
package main

import (
	"container/heap"
	"fmt"
)

type IntMinHeap []int

func (h IntMinHeap) Len() int {
	return len(h)
}

func (h IntMinHeap) Less(i, j int) bool {
	return h[i] < h[j]
}

func (h IntMinHeap) Swap(i, j int) {
	h[i], h[j] = h[j], h[i]
}

func (h *IntMinHeap) Push(x any) {
	*h = append(*h, x.(int))
}

func (h *IntMinHeap) Pop() any {
	old := *h
	n := len(old)
	x := old[n-1]
	*h = old[:n-1]
	return x
}

func TopK(nums []int, k int) []int {
	h := &IntMinHeap{}
	heap.Init(h)

	for _, x := range nums {
		if h.Len() < k {
			heap.Push(h, x)
			continue
		}

		if x > (*h)[0] {
			heap.Pop(h)
			heap.Push(h, x)
		}
	}

	return []int(*h)
}

func main() {
	nums := []int{3, 1, 5, 2, 9, 7, 8}

	fmt.Println(TopK(nums, 3))
}
```

---

## 【文本块 33】代码解释

这里用小根堆维护最大的 k 个数。

为什么是小根堆？

因为堆里保存的是当前最大的 k 个数。
其中最小的那个在堆顶。

当新元素比堆顶还大，说明它应该进入 TopK，就弹出堆顶，再插入新元素。

复杂度是：

```text
O(n log k)
```

比直接排序的：

```text
O(n log n)
```

更适合 k 远小于 n 的场景。

面试里可以说：

> TopK 问题常用大小为 k 的堆。求最大 K 个用小根堆，求最小 K 个用大根堆。

---

# 十六、哈希表

## 【文本块 34】Q：哈希表解决什么问题？

哈希表解决的是快速定位问题。

它通过 hash 函数把 key 映射到桶位置，从而实现平均 O(1) 的插入、删除、查找。

Go 里的 map 就是哈希表。

常见用途：

* 两数之和
* 去重
* 计数
* 缓存
* 映射关系
* 快速判断是否存在
* 图的邻接表
* 字符频次统计

哈希表的核心问题有三个：

第一，hash 冲突。
不同 key 可能映射到同一个桶。

第二，扩容。
元素太多时，冲突变多，需要扩容降低负载。

第三，key 必须可比较。
Go map 的 key 必须支持 == 和 !=。

---

## 【代码块 18】两数之和：map 记录补数

```go
package main

import "fmt"

func TwoSum(nums []int, target int) []int {
	indexMap := make(map[int]int)

	for i, x := range nums {
		need := target - x

		if j, ok := indexMap[need]; ok {
			return []int{j, i}
		}

		indexMap[x] = i
	}

	return nil
}

func main() {
	fmt.Println(TwoSum([]int{2, 7, 11, 15}, 9))
}
```

---

## 【文本块 35】树、堆、哈希怎么串起来回答？

可以按“解决什么问题”来组织：

树：表达层级和递归结构。
BST：解决有序查找。
平衡树：避免普通 BST 退化。
红黑树：工程里常见的平衡树实现。
B+ 树：数据库索引常用，适合磁盘 IO 和范围查询。
堆：快速拿最值，适合 TopK 和优先队列。
哈希表：快速精确定位，平均 O(1)。

面试里可以说：

> 如果只需要精确查找，哈希表通常更快；如果还需要有序、范围查询、前驱后继，树结构更合适；如果只关心最大值或最小值，堆更合适。

---

# 十七、排序算法

## 【文本块 36】Q：常见排序算法怎么分类？

可以按三种方式分类。

第一，按是否基于比较：

比较排序：

* 冒泡排序
* 选择排序
* 插入排序
* 希尔排序
* 快速排序
* 归并排序
* 堆排序

非比较排序：

* 计数排序
* 桶排序
* 基数排序

第二，按是否稳定：

稳定排序：

* 冒泡排序
* 插入排序
* 归并排序
* 计数排序
* 桶排序
* 基数排序

不稳定排序：

* 选择排序
* 快速排序
* 堆排序
* 希尔排序

第三，按是否原地：

原地排序：

* 冒泡
* 选择
* 插入
* 快排
* 堆排

非原地排序：

* 归并排序通常需要额外数组
* 计数、桶、基数也需要额外空间

面试重点通常是：

* 快排
* 归并
* 堆排
* 插入排序
* TopK
* 链表排序

---

## 【文本块 37】常见排序复杂度表

冒泡排序：平均 O(n²)，稳定，原地。
选择排序：平均 O(n²)，不稳定，原地。
插入排序：平均 O(n²)，稳定，原地，近乎有序时表现好。
快速排序：平均 O(n log n)，最坏 O(n²)，不稳定，原地。
归并排序：稳定，O(n log n)，需要 O(n) 额外空间。
堆排序：O(n log n)，不稳定，原地。
计数排序：O(n + k)，稳定，适合整数范围不大的场景。

面试里可以说：

> 排序题不要只背复杂度，还要说适用场景。快排平均性能好，归并稳定且适合链表排序，堆排适合原地和 TopK 思路，计数排序适合数据范围有限的整数。

---

## 【代码块 19】快速排序

```go
package main

import "fmt"

func QuickSort(nums []int) {
	var sort func(left, right int)

	sort = func(left, right int) {
		if left >= right {
			return
		}

		pivot := nums[left+(right-left)/2]
		i, j := left, right

		for i <= j {
			for nums[i] < pivot {
				i++
			}
			for nums[j] > pivot {
				j--
			}

			if i <= j {
				nums[i], nums[j] = nums[j], nums[i]
				i++
				j--
			}
		}

		sort(left, j)
		sort(i, right)
	}

	sort(0, len(nums)-1)
}

func main() {
	nums := []int{5, 2, 9, 1, 3, 7}

	QuickSort(nums)

	fmt.Println(nums)
}
```

---

## 【文本块 38】快排为什么平均快？

快速排序的核心是分治。

每次选择一个 pivot，把数组划分成两部分：

* 小于 pivot 的放左边
* 大于 pivot 的放右边

然后递归处理左右两边。

平均情况下，每次划分比较均衡，递归深度是 O(log n)，每层处理 O(n)，所以平均复杂度是 O(n log n)。

但如果 pivot 选得很差，比如每次都选到最小或最大值，递归会退化成 O(n)，总复杂度变成 O(n²)。

优化方式：

* 随机选 pivot
* 三数取中
* 小数组切换插入排序
* 控制递归深度
* 三路快排处理大量重复元素

---

## 【代码块 20】归并排序

```go
package main

import "fmt"

func MergeSort(nums []int) []int {
	if len(nums) <= 1 {
		return nums
	}

	mid := len(nums) / 2

	left := MergeSort(nums[:mid])
	right := MergeSort(nums[mid:])

	return Merge(left, right)
}

func Merge(left, right []int) []int {
	result := make([]int, 0, len(left)+len(right))

	i, j := 0, 0

	for i < len(left) && j < len(right) {
		if left[i] <= right[j] {
			result = append(result, left[i])
			i++
		} else {
			result = append(result, right[j])
			j++
		}
	}

	result = append(result, left[i:]...)
	result = append(result, right[j:]...)

	return result
}

func main() {
	nums := []int{5, 2, 9, 1, 3, 7}

	sorted := MergeSort(nums)

	fmt.Println(sorted)
}
```

---

## 【文本块 39】归并排序适合什么场景？

归并排序特点：

* 时间复杂度稳定 O(n log n)
* 稳定排序
* 适合链表排序
* 适合外部排序
* 缺点是需要额外空间

为什么适合链表？

因为链表本身不支持随机访问，快排依赖下标交换不方便。
归并只需要拆分和合并链表，能很好利用链表指针。

面试里可以说：

> 数组排序常用快排，链表排序常用归并。因为链表随机访问差，但合并两个有序链表很自然。

---

# 十八、栈与队列

## 【文本块 40】Q：栈和队列有什么区别？

栈是后进先出，LIFO。

常见操作：

* push
* pop
* top

适合：

* 括号匹配
* 表达式求值
* 单调栈
* DFS 显式栈
* 函数调用栈
* 回溯状态

队列是先进先出，FIFO。

常见操作：

* offer
* poll
* front

适合：

* BFS
* 任务调度
* 消息队列
* 滑动窗口
* 层序遍历
* 生产者消费者

Go 里没有内置 stack 类型，通常用 slice 实现。

---

## 【代码块 21】Go 用 slice 实现栈

```go
package main

import "fmt"

type Stack[T any] struct {
	items []T
}

func (s *Stack[T]) Push(x T) {
	s.items = append(s.items, x)
}

func (s *Stack[T]) Pop() (T, bool) {
	var zero T

	if len(s.items) == 0 {
		return zero, false
	}

	n := len(s.items)
	x := s.items[n-1]
	s.items = s.items[:n-1]

	return x, true
}

func (s *Stack[T]) Peek() (T, bool) {
	var zero T

	if len(s.items) == 0 {
		return zero, false
	}

	return s.items[len(s.items)-1], true
}

func main() {
	var st Stack[int]

	st.Push(1)
	st.Push(2)

	x, _ := st.Pop()
	fmt.Println(x)
}
```

---

## 【代码块 22】Go 用 slice 实现队列

```go
package main

import "fmt"

type Queue[T any] struct {
	items []T
	head  int
}

func (q *Queue[T]) Push(x T) {
	q.items = append(q.items, x)
}

func (q *Queue[T]) Pop() (T, bool) {
	var zero T

	if q.head >= len(q.items) {
		return zero, false
	}

	x := q.items[q.head]
	q.head++

	if q.head > 1024 && q.head*2 > len(q.items) {
		q.items = append([]T(nil), q.items[q.head:]...)
		q.head = 0
	}

	return x, true
}

func main() {
	var q Queue[int]

	q.Push(1)
	q.Push(2)

	x, _ := q.Pop()
	fmt.Println(x)
}
```

---

## 【文本块 41】为什么队列不建议每次 nums = nums[1:]？

用 slice 做队列时，很多人会写：

```go
queue = queue[1:]
```

这本身没错，但要注意两个问题。

第一，频繁截取可能让底层数组长期被引用，导致已经出队的元素无法及时释放。
第二，如果元素是指针或大对象引用，老位置仍然可能影响 GC。

所以更稳的写法是用 head 指针记录队头。
当 head 太大时，再整体拷贝剩余元素，释放前面的空间。

力扣里数据量不大时，`queue = queue[1:]` 通常可以接受。
工程代码里要更谨慎。

---

# 十九、单调栈

## 【文本块 42】Q：什么是单调栈？

单调栈不是新的数据结构，而是带约束的栈。

它要求栈内元素保持单调递增或单调递减。

它最常用于解决：

> 找每个元素左边或右边第一个比它大/小的元素。

典型题：

* 下一个更大元素
* 每日温度
* 柱状图最大矩形
* 接雨水
* 去除重复字母

单调栈的核心是：

> 当新元素进入时，把栈顶所有不满足单调性的元素弹出。

被弹出的那一刻，通常就找到了它的答案。

---

## 【代码块 23】每日温度：单调栈

```go
package main

import "fmt"

func DailyTemperatures(temperatures []int) []int {
	n := len(temperatures)
	ans := make([]int, n)
	stack := make([]int, 0)

	for i, t := range temperatures {
		for len(stack) > 0 && temperatures[stack[len(stack)-1]] < t {
			j := stack[len(stack)-1]
			stack = stack[:len(stack)-1]
			ans[j] = i - j
		}

		stack = append(stack, i)
	}

	return ans
}

func main() {
	fmt.Println(DailyTemperatures([]int{73, 74, 75, 71, 69, 72, 76, 73}))
}
```

---

## 【文本块 43】代码解释

栈里保存的是下标，而不是温度值。

并且栈对应的温度保持单调递减。

当遇到一个更高温度 t 时，说明栈顶那些比 t 小的日子都找到了“下一个更暖和的日子”。

所以不断弹栈并计算距离。

每个元素最多入栈一次、出栈一次，所以总复杂度是 O(n)。

面试里可以说：

> 单调栈表面有 while，但每个元素只会被压入和弹出一次，所以整体是线性复杂度。

---

# 二十、单调队列

## 【文本块 44】Q：什么是单调队列？

单调队列通常用于滑动窗口最值问题。

它维护一个双端队列，使队列中的元素保持单调。

以滑动窗口最大值为例：

* 队头始终是当前窗口最大值
* 新元素进入时，把队尾所有比它小的元素弹出
* 旧元素滑出窗口时，如果它正好在队头，就弹出队头

典型题：

* 滑动窗口最大值
* 队列最大值
* 带窗口限制的动态规划优化

---

## 【代码块 24】滑动窗口最大值：单调队列

```go
package main

import "fmt"

func MaxSlidingWindow(nums []int, k int) []int {
	if k == 0 || len(nums) == 0 {
		return nil
	}

	deque := make([]int, 0)
	ans := make([]int, 0)

	for i, x := range nums {
		for len(deque) > 0 && nums[deque[len(deque)-1]] <= x {
			deque = deque[:len(deque)-1]
		}

		deque = append(deque, i)

		if deque[0] <= i-k {
			deque = deque[1:]
		}

		if i >= k-1 {
			ans = append(ans, nums[deque[0]])
		}
	}

	return ans
}

func main() {
	fmt.Println(MaxSlidingWindow([]int{1, 3, -1, -3, 5, 3, 6, 7}, 3))
}
```

---

## 【文本块 45】单调栈和单调队列怎么区分？

单调栈常用于：

> 找某个元素旁边第一个更大或更小的元素。

比如每日温度、下一个更大元素。

单调队列常用于：

> 维护一个滑动窗口内的最大值或最小值。

比如滑动窗口最大值。

一句话记忆：

> 单调栈解决“最近更大/更小”，单调队列解决“窗口最值”。

---

# 二十一、Trie 和并查集

## 【文本块 46】Q：Trie 和并查集要不要会？

要有基本概念。

它们不是每场面试必问，但在特定题型里非常好用。

Trie，又叫前缀树。
适合：

* 字符串前缀匹配
* 搜索推荐
* 敏感词过滤
* 字典树
* 单词搜索

并查集适合：

* 连通性问题
* 判断两个节点是否属于同一集合
* 合并集合
* 岛屿数量
* 朋友圈
* 冗余连接
* Kruskal 最小生成树

面试里可以说：

> 如果题目出现前缀、词典、搜索建议，可以考虑 Trie；如果题目出现连通块、合并集合、关系传递，可以考虑并查集。

---

## 【代码块 25】并查集模板

```go
package main

import "fmt"

type UnionFind struct {
	parent []int
	size   []int
}

func NewUnionFind(n int) *UnionFind {
	parent := make([]int, n)
	size := make([]int, n)

	for i := 0; i < n; i++ {
		parent[i] = i
		size[i] = 1
	}

	return &UnionFind{
		parent: parent,
		size:   size,
	}
}

func (uf *UnionFind) Find(x int) int {
	if uf.parent[x] != x {
		uf.parent[x] = uf.Find(uf.parent[x])
	}
	return uf.parent[x]
}

func (uf *UnionFind) Union(a, b int) {
	rootA := uf.Find(a)
	rootB := uf.Find(b)

	if rootA == rootB {
		return
	}

	if uf.size[rootA] < uf.size[rootB] {
		rootA, rootB = rootB, rootA
	}

	uf.parent[rootB] = rootA
	uf.size[rootA] += uf.size[rootB]
}

func (uf *UnionFind) Connected(a, b int) bool {
	return uf.Find(a) == uf.Find(b)
}

func main() {
	uf := NewUnionFind(5)

	uf.Union(0, 1)
	uf.Union(1, 2)

	fmt.Println(uf.Connected(0, 2))
	fmt.Println(uf.Connected(0, 4))
}
```

---

## 【文本块 47】并查集为什么快？

并查集有两个核心优化：

第一，路径压缩。
Find 的时候，把沿途节点直接挂到根节点下面。

第二，按大小或秩合并。
Union 的时候，把小树挂到大树下面，避免树太高。

有了这两个优化后，并查集的单次操作复杂度可以近似看作 O(1)。

面试里不用展开反阿克曼函数，只要说：

> 路径压缩 + 按秩合并后，并查集操作几乎是常数级。

---

# 二十二、Go 算法题常用工具

## 【文本块 48】Go 刷题常用标准库

第一，sort 包。

常见方法：

```go
sort.Ints(nums)
sort.Strings(strs)
sort.Slice(items, func(i, j int) bool { ... })
sort.Search(n, func(i int) bool { ... })
```

第二，container/heap。

用于优先队列和 TopK。

第三，math 包。

比如：

```go
math.MaxInt
math.MinInt
```

第四，strings 包。

用于字符串处理。

第五，strconv 包。

字符串和数字转换。

第六，unicode 包。

判断字符类别。

第七，container/list。

双向链表。业务里用得不多，某些 LRU 实现会用。

---

## 【代码块 26】sort.Search 写二分

```go
package main

import (
	"fmt"
	"sort"
)

func LowerBound(nums []int, target int) int {
	return sort.Search(len(nums), func(i int) bool {
		return nums[i] >= target
	})
}

func main() {
	nums := []int{1, 2, 2, 2, 3, 4}

	fmt.Println(LowerBound(nums, 2))
	fmt.Println(LowerBound(nums, 5))
}
```

---

## 【文本块 49】sort.Search 怎么理解？

`sort.Search(n, f)` 会在 `[0, n)` 里找第一个让 f(i) 为 true 的位置。

前提是 f 具有单调性：

```text
false false false true true true
```

所以它天然适合找左边界和二分答案。

面试里可以说：

> Go 标准库 sort.Search 本质就是二分左边界模板。只要能把问题转成“找第一个满足条件的位置”，就可以用它。

---

# 二十三、数据结构算法速记版

## 【文本块 50】复杂度速记

复杂度看的是增长趋势，不是某次运行的毫秒数。

O(1)：数组下标访问、map 平均查找。
O(log n)：二分查找、堆插入删除、平衡树查找。
O(n)：遍历数组、链表查找。
O(n log n)：常见高效排序。
O(n²)：双层循环、简单排序。

工程性能还要看常数、内存分配、缓存局部性、锁竞争和 IO。

---

## 【文本块 51】数组链表速记

数组和 slice：

* 连续内存
* 随机访问 O(1)
* 遍历快
* 中间插删 O(n)
* Go 里最常用的是 slice

链表：

* 节点分散
* 不支持随机访问
* 查找 O(n)
* 已知节点插删 O(1)
* 缓存局部性差
* 力扣题常见，业务代码相对少

链表题套路：

* 反转
* 快慢指针
* dummy 虚拟头
* 删除倒数第 k 个
* 找中点
* 判断环
* 合并链表

---

## 【文本块 52】树堆哈希速记

树：

* 表达层级关系
* 二叉树遍历是基础
* BST 中序有序
* DFS 适合路径和递归
* BFS 适合层序和最短路径

堆：

* 完全二叉树
* 快速拿最大值或最小值
* 插入删除 O(log n)
* TopK 用堆
* Go 用 container/heap

哈希：

* 快速定位
* 平均 O(1)
* Go 用 map
* 适合去重、计数、映射、补数问题
* 需要注意 key 可比较、冲突、扩容、并发安全

一句话：

> 树解决层级和有序，堆解决最值，哈希解决快速定位。

---

## 【文本块 53】排序速记

快排：

* 平均 O(n log n)
* 最坏 O(n²)
* 不稳定
* 实际常用
* pivot 选择很关键

归并：

* 稳定
* O(n log n)
* 需要额外空间
* 适合链表排序和外部排序

堆排：

* O(n log n)
* 原地
* 不稳定
* 和 TopK 思路相关

插入排序：

* O(n²)
* 稳定
* 近乎有序时表现好

计数排序：

* 非比较排序
* 适合整数范围不大的场景

---

## 【文本块 54】栈队列速记

栈：

* 后进先出
* 括号匹配
* 单调栈
* DFS
* 递归调用栈
* 表达式求值

队列：

* 先进先出
* BFS
* 层序遍历
* 任务调度
* 生产者消费者

单调栈：

* 找最近更大或更小
* 每个元素最多入栈出栈一次
* 整体 O(n)

单调队列：

* 维护滑动窗口最大值或最小值
* 队头是窗口最值
* 整体 O(n)

---

# 二十四、本章最终面试回答模板

## 【文本块 55】设计模式综合模板

如果面试官问我 Go 里怎么理解设计模式，我会这样回答：

Go 里当然有设计模式，但表达方式和 Java 不太一样。Java 常常通过类继承、抽象类、接口层次来实现模式，而 Go 更强调小接口、组合、函数类型和显式依赖。比如策略模式在 Go 里可以用 interface，也可以直接用函数类型；单例模式一般用 sync.Once；观察者模式可以用本地事件总线，跨服务时通常演进成 MQ；装饰器模式在 Go 里最常见的形式是 HTTP middleware 或 gRPC interceptor；对象构造复杂时可以用 Functional Options。

我理解设计模式不是背类图，而是看它解决什么问题。策略模式解决可替换算法，单例解决一次初始化，观察者解决事件通知和解耦，适配器解决第三方接口不一致，装饰器解决不改原逻辑增强能力，依赖注入解决依赖关系显式化和可测试性。Go 的工程风格是少继承、多组合、少全局状态、多构造注入，能用简单函数解决就不要过度抽象。

---

## 【文本块 56】数据结构算法综合模板

如果面试官问我数据结构和算法基础，我会这样回答：

我会先从复杂度说起。时间复杂度和空间复杂度描述的是输入规模增长时，算法执行时间和额外空间的增长趋势。常见的有 O(1)、O(log n)、O(n)、O(n log n)、O(n²)。但工程里还要考虑常数、缓存局部性、内存分配和 IO。

在线性结构里，数组和 Go 的 slice 是连续内存，随机访问快、遍历快，但中间插删需要移动元素；链表节点不连续，已知节点插删快，但查找慢，缓存局部性也差。所以 Go 业务里更常用 slice，链表更多出现在算法题里。链表题高频套路是反转、快慢指针、dummy 节点、找中点、找环、删除倒数第 k 个。

树、堆、哈希分别解决不同问题。树适合表达层级关系和有序关系，二叉树重点是 DFS、BFS、前中后序和层序遍历；BST 的中序遍历是有序的。堆适合快速拿最大值或最小值，常用于 TopK、优先队列、合并 K 个有序链表。哈希表适合快速定位，Go 里就是 map，常用于去重、计数、补数、映射关系，平均查找是 O(1)，但要注意 key 必须可比较，且并发读写不安全。

排序算法里，快排平均 O(n log n)，实际常用但最坏会退化；归并排序稳定，适合链表排序和外部排序；堆排序 O(n log n)，原地但不稳定；计数排序适合整数范围有限的场景。栈是后进先出，常用于括号匹配、单调栈、DFS；队列是先进先出，常用于 BFS、层序遍历、任务调度。单调栈解决最近更大或更小，单调队列解决滑动窗口最值。

一句话总结：数据结构面试不能只背定义，要能说清楚每种结构解决什么问题、复杂度是什么、Go 里怎么实现、典型题型是什么，以及和相邻结构怎么区分。
